# Lab 2 - Part 1: CNN, Faster R-CNN and Transfer Learning

Dataset: https://www.kaggle.com/datasets/hojjatk/mnist-dataset

This part compares fully connected, convolutional, detection and pretrained image models on MNIST.


In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torch', 'torchvision', 'scikit-learn'])

In [ ]:
from pathlib import Path
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path
from importlib.util import spec_from_file_location, module_from_spec
script = Path('mnist_gpu_lab.py')
spec = spec_from_file_location('mnist_gpu_lab', script)
lab = module_from_spec(spec)
spec.loader.exec_module(lab)
train_loader, test_loader = lab.loaders('./data', batch_size=128, limit=12000)
transfer_train_loader, transfer_test_loader = lab.transfer_loaders('./data', batch_size=32, limit=12000)
print('Device:', lab.device())


## 1. FCNN and CNN classifiers

The FCNN is a dense baseline. The CNN uses convolution, pooling, dropout and fully connected layers.


In [ ]:
results = {}
for name, model in [('FCNN', lab.FCNN()), ('CNN', lab.CNN())]:
    results[name] = lab.train(model, train_loader, test_loader, epochs=2, lr=1e-3)
    print(name, results[name])


## 2. Faster R-CNN

MNIST has no native bounding boxes, so this demonstration uses one synthetic full-image box per digit.


In [ ]:
detector = lab.faster_rcnn_for_mnist()
detector_results = lab.train_faster_rcnn(detector, lab.detection_loader('./data', batch_size=8, limit=256), epochs=1)
results['Faster R-CNN (synthetic boxes)'] = detector_results
print('Faster R-CNN (synthetic boxes)', detector_results)


## 3. VGG16 and AlexNet fine tuning

MNIST is converted to three channels and resized to 224x224 for ImageNet-style pretrained networks.


In [ ]:
for name in ['vgg16', 'alexnet']:
    results[name] = lab.train(lab.transfer_model(name), transfer_train_loader, transfer_test_loader, epochs=1, lr=1e-4)
    print(name, results[name])


In [ ]:
import pandas as pd
comparison_part1 = pd.DataFrame([
    {'Model': name, 'Loss': value.loss, 'Accuracy': value.accuracy,
     'Macro F1': value.macro_f1, 'Training time (s)': value.seconds}
    for name, value in results.items()
])
comparison_part1.sort_values('Accuracy', ascending=False)


## Interpretation

The CNN is expected to give the best balance between accuracy and computation for native MNIST. Faster R-CNN is only a demonstration because the original dataset has no real boxes. VGG16 and AlexNet show transfer learning but add unnecessary size and resizing cost for this simple dataset.
